## Score Resume Evidence to Market

Yes. This is the right breakdown.

The big correction is:

```text
canonical_master_resume.json = source of truth
candidate_evidence.json = extracted scoring units
```

You are not replacing the canonical resume. You are creating a smaller, cleaner artifact for Stage 4.

---

## 1. `candidate_evidence.json`

### What it does

It converts the canonical resume into discrete pieces of evidence that can be scored.

The canonical resume contains:

```text
roles
companies
dates
titles
bullets
projects
archive facts
education
certifications
awards
patents
teaching
```

But Stage 4 does not need to reason over the whole resume object.

It needs scoring units like:

```json
{
  "evidence_id": "coned_001",
  "source_type": "experience_bullet",
  "role_id": "abc_consulting_coned",
  "company": "ABC Consulting / ConEd",
  "date_range": "2026-present",
  "evidence_text": "Built a guardrailed analytics assistant using Gemini, validated JSON instructions, approved tools, SQL generation, and BigQuery-backed visualizations.",
  "context": "ConEd anomaly detection and analytics POC",
  "recency_weight_hint": "high"
}
```

### Why this is better than scoring `master_resume.json`

Because the canonical resume is too rich and uneven.

Some things are bullets. Some are role summaries. Some are archive notes. Some are projects. Some are credentials.

`candidate_evidence.json` normalizes them into one comparable shape:

```text
one accomplishment / fact / project / proof point per record
```

That makes scoring easier, more auditable, and less prompt-chaotic.

### How to build it

Mostly deterministic extraction.

Start with:

```text
experience.roles[].bullets[]
projects[]
patents[]
certifications[]
teaching[]
selected archive facts, if available
```

Each extracted item becomes one evidence record.

Do **not** score yet.

Do **not** align to the archetype yet.

Do **not** rewrite aggressively.

Just extract and lightly normalize.

Recommended fields:

```json
{
  "evidence_id": "string",
  "source_type": "experience_bullet | project | patent | certification | teaching | award | archive_fact",
  "source_id": "string",
  "company": "string",
  "role": "string",
  "date_range": "string",
  "evidence_text": "string",
  "context": "string",
  "raw_text": "string",
  "confidence": "high | medium | low"
}
```

Key rule:

```text
candidate_evidence.json should contain candidate facts only.
No market interpretation.
```

---

## 2. Signal → Evidence Map

This is the first alignment artifact.

Input:

```text
candidate_evidence.json
target_archetype.json
canonical_market_signals.json
```

Output:

```text
signal_evidence_map.json
```

Purpose:

For each target signal, identify which evidence supports it.

Example:

```json
{
  "signal": "Productionizing generative/agentic AI",
  "signal_importance": "must_emphasize",
  "supporting_evidence": [
    {
      "evidence_id": "coned_001",
      "relevance": 5,
      "reason": "Directly demonstrates a deployed-style GenAI analytics workflow with guardrails, tool routing, SQL generation, and validation."
    },
    {
      "evidence_id": "nbcu_002",
      "relevance": 3,
      "reason": "Shows agentic architecture exploration, but engagement duration was short and production depth is limited."
    }
  ]
}
```

This answers:

```text
For each thing the market wants, what proof do we have?
```

This is useful because it exposes gaps.

Example:

```text
AI Governance → strong evidence
Agentic AI → strong evidence
Cloud-native architecture → moderate evidence
Formal people management → weak evidence
```

That is valuable before generating anything.

---

## 3. Evidence Scoring

This is the reverse view.

Input:

```text
candidate_evidence.json
target_archetype.json
canonical_market_signals.json
```

Output:

```text
evidence_scores.json
```

Purpose:

For each evidence item, score how strongly it supports important target signals.

Example:

```json
{
  "evidence_id": "coned_001",
  "evidence_text": "Built a guardrailed analytics assistant using Gemini...",
  "signal_scores": [
    {
      "signal": "Productionizing generative/agentic AI",
      "score": 5,
      "reason": "Direct implementation of GenAI workflow with tool selection, validation, and user-facing analytics."
    },
    {
      "signal": "MLOps/LLMOps/AgentOps leadership and operationalization",
      "score": 4,
      "reason": "Shows guardrails, validation, reproducibility, and controlled execution, though not full lifecycle production operations."
    },
    {
      "signal": "Responsible AI: governance, compliance, security, and explainability",
      "score": 4,
      "reason": "Uses explicit guardrails and controlled SQL/tool execution in a regulated utility context."
    }
  ],
  "overall_score": 4.4,
  "recommendation": "must_include"
}
```

This answers:

```text
Which accomplishments deserve resume space?
```

Important distinction:

```text
signal_evidence_map.json = market-first view
evidence_scores.json = resume-selection view
```

You probably want both.

---

## Signal Importance

I would not overcomplicate this yet.

Use simple tiers from Stage 3:

```text
must_emphasize = weight 3
should_include = weight 2
nice_to_have = weight 1
```

Then calculate:

```text
overall_score =
sum(signal_score * signal_weight) / sum(signal_weight)
```

Later you can improve it, but this is enough for Phase 4.

---

## Phase 4 Completion Definition

Phase 4 is complete when you have:

```text
candidate_evidence.json
signal_evidence_map.json
evidence_scores.json
selected_evidence.json
```

And can say:

```text
We extracted candidate evidence from the canonical resume.
We aligned each evidence item to market-derived signals.
We scored evidence using Stage 3 signals, not hardcoded assumptions.
We selected the evidence that best supports the target archetype.
```

That is clean, bounded, and not spiraling.


### 4a - gen candidate_evidence.json from canonical resume

In [1]:
from pathlib import Path
import json
import pandas as pd
from src.config import ARTIFACT_DIR
from dotenv import load_dotenv
import json
from src.helpers import load_json, save_json
from collections import Counter

BUILD_INITIAL_TARGET_RESUME = False

canonical_resume = load_json(ARTIFACT_DIR / "canonical_master_resume.json")

In [2]:
def get_sections_by_type(canonical_resume, section_type):
    return [
        s for s in canonical_resume.get("sections", [])
        if s.get("type") == section_type
    ]

def extract_candidate_evidence(canonical_resume):
    evidence = []

    experience_sections = get_sections_by_type(canonical_resume, "experience")

    for section_idx, section in enumerate(experience_sections, start=1):
        section_heading = section.get("heading", "Experience")

        for exp_idx, exp in enumerate(section.get("content", []), start=1):
            role = exp.get("role", "")
            organization = exp.get("organization", "")
            exp_dates = exp.get("dates", "")
            source_label = " | ".join(
                x for x in [organization, role] if x)

            for block_idx, block in enumerate(exp.get("content", []), start=1):
                # Direct bullet string
                if isinstance(block, str):
                    evidence.append({
                        "evidence_id": f"exp_{section_idx:02d}_{exp_idx:02d}_{block_idx:02d}_01",
                        "source_label": source_label,
                        "source_type": "experience_bullet",
                        "source_section": section_heading,
                        "organization": organization,
                        "role": role,
                        "dates": exp_dates,
                        "label": "",
                        "context": "",
                        "evidence_text": block,
                        "raw_text": block,
                        "confidence": "high"
                    })
                    continue

                # Subsection containing bullets
                if isinstance(block, dict):
                    label = block.get("label", "")
                    context = block.get("context", "")
                    block_dates = block.get("dates", exp_dates)
                    block_type = block.get("type", "")
                    source_label = " | ".join(
                        x for x in [organization, role, label] if x)

                    if block_type == "bullet":
                        for bullet_idx, bullet in enumerate(block.get("content", []), start=1):
                            evidence.append({
                                "evidence_id": f"exp_{section_idx:02d}_{exp_idx:02d}_{block_idx:02d}_{bullet_idx:02d}",
                                "source_label": source_label,
                                "source_type": "experience_bullet",
                                "source_section": section_heading,
                                "organization": organization,
                                "role": role,
                                "dates": block_dates,
                                "label": label,
                                "context": context,
                                "evidence_text": bullet,
                                "raw_text": bullet,
                                "confidence": "high"
                            })

    return evidence

In [3]:
candidate_evidence = extract_candidate_evidence(canonical_resume)

In [4]:
# Look at the distribution of jobs to evidence -- hopefully distributed across jobs.
Counter(e["source_label"] for e in candidate_evidence)

Counter({'Capital One | Senior Manager, Operations Analysis': 6,
         'Meta | Data Scientist': 5,
         'Facteus | Head of Data Science': 5,
         'Daly Engineers LLC | Principal Consultant | Consolidated Edison (via ABC Consulting)': 4,
         'Daly Engineers LLC | Principal Consultant | NBCUniversal (via Apex Systems)': 4,
         'SimpliLearn | AI Instructor': 4,
         'Intuit (via The Cydio Group) | Data Scientist': 4,
         'Samsung Electronics (via Harvey Nash) | Data Scientist': 4,
         'Raytheon | Senior Principal Systems Engineer / Program Manager': 4,
         'Interview Kickstart | AI Instructor': 3,
         'Nike (via Intersoft Inc.) | Expert SEO Solutions Architect': 3})

In [5]:
save_json(candidate_evidence, ARTIFACT_DIR/ "candidate_evidence.json")

### Phase 4B: Map Evidence to Market Signals
Goal

Create:

artifacts/signal_evidence_map.json

Question answered:

For each market-derived signal, which candidate evidence supports it, and how directly?

Recommended Contract
```json
[
  {
    "signal": "Productionizing generative/agentic AI",
    "signal_priority": "must_emphasize",
    "supporting_evidence": [
      {
        "evidence_id": "exp_01_01_01_04",
        "relevance": 4,
        "support_type": "direct",
        "reason": "Shows implementation of a guardrailed GenAI analytics workflow with controlled tool use and BigQuery-backed execution."
      }
    ],
    "coverage_assessment": "strong"
  }
]
```

Load Stage 3 signals -- job archetype

In [6]:
from pathlib import Path
import json
from typing import Any
from src.capabilities import call_json_model

from langchain_openai import ChatOpenAI

load_dotenv()
model = ChatOpenAI(
    model="gpt-4.1",
    temperature=0,
)


candidate_evidence = load_json(ARTIFACT_DIR / "candidate_evidence.json")
target_archetype = load_json(ARTIFACT_DIR / "target_archetype.json")

In [7]:
# categorize keys in the target archetype
signal_source_priority = {
        "core_market_themes": 3,
        "target_capabilities": 3,
        "target_problem_spaces": 2,
        "target_technology_areas": 2,
        "seniority_expectations": 3,
        "success_patterns": 2,
    }

CONTEXT_FIELDS = [
    "title",
    "archetype_summary",
    "market_basis",
    "hypothesis_assessment",
    "resume_implications",
    "archetype_narrative",
]

In [8]:
def normalize_signal_items(value):
    """
    Converts a target_archetype field into a list of signal strings.

    Handles:
    - list[str]
    - dict[str, list[str]]
    - dict[str, str]
    - str
    """
    signals = []

    if value is None:
        return signals

    if isinstance(value, str):
        return [value]

    if isinstance(value, list):
        for item in value:
            if isinstance(item, str):
                signals.append(item)
            elif isinstance(item, dict):
                for k, v in item.items():
                    if isinstance(v, str):
                        signals.append(f"{k}: {v}")
                    elif isinstance(v, list):
                        for x in v:
                            signals.append(f"{k}: {x}")
            else:
                signals.append(str(item))
        return signals

    if isinstance(value, dict):
        for k, v in value.items():
            if isinstance(v, str):
                signals.append(f"{k}: {v}")
            elif isinstance(v, list):
                for item in v:
                    signals.append(f"{k}: {item}")
            elif isinstance(v, dict):
                for kk, vv in v.items():
                    signals.append(f"{k} / {kk}: {vv}")
            else:
                signals.append(f"{k}: {v}")
        return signals

    return [str(value)]

In [9]:
def extract_target_signals(target_archetype):

    target_signals = []

    for field in signal_source_priority.keys():
        raw_value = target_archetype.get(field)
        signals = normalize_signal_items(raw_value)

        for idx, signal in enumerate(signals, start=1):
            target_signals.append({
                "signal_id": f"{field}_{idx:03d}",
                "signal": signal,
                "source_field": field,
                "priority_weight": signal_source_priority[field],
            })

    return target_signals

In [10]:
target_signals = extract_target_signals(target_archetype)

len(target_signals), target_signals[:10]

(52,
 [{'signal_id': 'core_market_themes_001',
   'signal': 'Enterprise-scale AI/ML platform architecture and delivery',
   'source_field': 'core_market_themes',
   'priority_weight': 3},
  {'signal_id': 'core_market_themes_002',
   'signal': 'Productionization and operationalization of generative and agentic AI',
   'source_field': 'core_market_themes',
   'priority_weight': 3},
  {'signal_id': 'core_market_themes_003',
   'signal': 'Cloud-native, scalable, and resilient system design',
   'source_field': 'core_market_themes',
   'priority_weight': 3},
  {'signal_id': 'core_market_themes_004',
   'signal': 'MLOps, LLMOps, and observability as operational foundations',
   'source_field': 'core_market_themes',
   'priority_weight': 3},
  {'signal_id': 'core_market_themes_005',
   'signal': 'Governance, compliance, and responsible AI as non-negotiable requirements',
   'source_field': 'core_market_themes',
   'priority_weight': 3},
  {'signal_id': 'core_market_themes_006',
   'signal': '

In [11]:
def compact_evidence_for_prompt(candidate_evidence):
    return [
        {
            "evidence_id": e["evidence_id"],
            "source_label": e.get("source_label", ""),
            "evidence_text": e["evidence_text"],
        }
        for e in candidate_evidence
    ]

compact_evidence = compact_evidence_for_prompt(candidate_evidence)

len(compact_evidence), compact_evidence[:2]

(46,
 [{'evidence_id': 'exp_01_01_01_01',
   'source_label': 'Daly Engineers LLC | Principal Consultant | Consolidated Edison (via ABC Consulting)',
   'evidence_text': 'Led analytical readiness assessment for Oracle/EBS and Maximo data migrated to BigQuery, establishing the foundation for cross-domain analytics across construction operations and financial systems.'},
  {'evidence_id': 'exp_01_01_01_02',
   'source_label': 'Daly Engineers LLC | Principal Consultant | Consolidated Edison (via ABC Consulting)',
   'evidence_text': 'Developed automated profiling, type inference, duplicate detection, candidate key analysis, and cross-system join validation frameworks across 100+ source tables and millions of operational and financial records.'}])

In [12]:
def build_signal_evidence_mapping_prompt(signal_spec, compact_evidence):
    expected = {
        "signal_id": signal_spec["signal_id"],
        "signal": signal_spec["signal"],
        "source_field": signal_spec["source_field"],
        "priority_weight": signal_spec["priority_weight"],
        "supporting_evidence": [
            {
                "evidence_id": "string",
                "relevance": 0,
                "support_type": "direct | partial | adjacent",
                "reason": "string"
            }
        ],
        "coverage_assessment": "strong | moderate | weak | unsupported",
        "coverage_reason": "string"
    }

    return f"""
You are mapping candidate evidence to a market-derived target signal.

This is evidence mapping, not resume writing.

Target signal:
{json.dumps(signal_spec, indent=2)}

Candidate evidence:
{json.dumps(compact_evidence, indent=2)}

Task:
Identify candidate evidence that meaningfully supports the target signal.

General rules:
- Use only the provided candidate evidence.
- Use evidence_id values exactly as provided. Do not invent, modify, or infer evidence IDs.
- Do not rewrite bullets.
- Do not create new claims.
- Do not infer accomplishments that are not explicitly supported.
- Do not reward keyword overlap by itself.
- Be conservative.
- It is acceptable to return no supporting evidence.
- Return only genuinely meaningful matches.
- Return at most 5 supporting evidence items.
- Most signals should have 0-3 matches.
- Never fill the list to reach the maximum.
- Include only evidence with relevance >= 3.
- Exclude weak keyword overlap.

Evidence type rules:
- Teaching evidence can support mentorship, communication, technical fluency, and developer enablement.
- Teaching evidence must not be treated as direct evidence of production deployment or operational ownership unless the evidence explicitly says the candidate built or deployed the system.
- Roadmaps, assessments, and evaluations are strategy evidence. They are not direct delivery evidence unless implementation is explicitly stated.
- Architecture or platform evidence is direct only when the evidence describes building, designing, deploying, or owning a system, framework, or platform.
- Governance evidence is direct only when the evidence describes controls, review processes, compliance, auditability, policy enforcement, or responsible AI implementation.
- Teaching governance is direct for governance education, but only partial for production governance implementation.
- Cloud-native evidence is direct only when the evidence names cloud platforms, infrastructure automation, containerized workloads, managed services, or scalable deployment patterns.
- Do not include adjacent evidence unless there are fewer than 3 stronger matches and the adjacent evidence adds useful coverage.

Relevance scale:
5 = direct, unusually strong evidence
4 = direct and credible evidence
3 = relevant but incomplete
2 = adjacent or partial support, exclude unless explicitly needed
1 = weak keyword overlap only, exclude
0 = no meaningful support, exclude

Support type values:
- direct
- partial
- adjacent

Coverage assessment:
- strong = at least 3 direct evidence items, including some high-relevance matches
- moderate = 1-2 direct items or several partial items
- weak = limited adjacent support only
- unsupported = no meaningful support

Return valid JSON only.

Return exactly this structure:
{json.dumps(expected, indent=2)}
"""

In [13]:
def validate_signal_mapping_result(result, signal_spec, valid_evidence_ids):
    errors = []

    required_keys = {
        "signal_id",
        "signal",
        "source_field",
        "priority_weight",
        "supporting_evidence",
        "coverage_assessment",
        "coverage_reason",
    }

    missing = required_keys - set(result.keys())
    if missing:
        errors.append(f"Missing keys: {sorted(missing)}")

    if result.get("signal_id") != signal_spec["signal_id"]:
        errors.append(
            f"signal_id mismatch: expected {signal_spec['signal_id']}, got {result.get('signal_id')}"
        )

    if result.get("signal") != signal_spec["signal"]:
        errors.append("signal text mismatch")

    allowed_coverage = {"strong", "moderate", "weak", "unsupported"}
    if result.get("coverage_assessment") not in allowed_coverage:
        errors.append(f"Invalid coverage_assessment: {result.get('coverage_assessment')}")

    allowed_support_types = {"direct", "partial", "adjacent", "weak"}

    for idx, item in enumerate(result.get("supporting_evidence", [])):
        evidence_id = item.get("evidence_id")
        relevance = item.get("relevance")
        support_type = item.get("support_type")

        if evidence_id not in valid_evidence_ids:
            errors.append(f"Unknown evidence_id at supporting_evidence[{idx}]: {evidence_id}")

        if not isinstance(relevance, int) or not (2 <= relevance <= 5):
            errors.append(f"Invalid relevance at supporting_evidence[{idx}]: {relevance}")

        if support_type not in allowed_support_types:
            errors.append(f"Invalid support_type at supporting_evidence[{idx}]: {support_type}")

    return errors

In [14]:
valid_evidence_ids = {e["evidence_id"] for e in candidate_evidence}

signal_evidence_map = []
mapping_errors = []

for i, signal_spec in enumerate(target_signals, start=1):
    print(f"{i}/{len(target_signals)}: {signal_spec['signal']}")

    prompt = build_signal_evidence_mapping_prompt(
        signal_spec=signal_spec,
        compact_evidence=compact_evidence,
    )

    result = call_json_model(prompt, model)

    errors = validate_signal_mapping_result(
        result=result,
        signal_spec=signal_spec,
        valid_evidence_ids=valid_evidence_ids,
    )

    if errors:
        mapping_errors.append({
            "signal_id": signal_spec["signal_id"],
            "signal": signal_spec["signal"],
            "errors": errors,
            "raw_result": result,
        })

    signal_evidence_map.append(result)

1/52: Enterprise-scale AI/ML platform architecture and delivery
2/52: Productionization and operationalization of generative and agentic AI
3/52: Cloud-native, scalable, and resilient system design
4/52: MLOps, LLMOps, and observability as operational foundations
5/52: Governance, compliance, and responsible AI as non-negotiable requirements
6/52: Mentorship, cross-functional leadership, and technical influence
7/52: Continuous innovation, research application, and developer enablement
8/52: Architecting and designing scalable AI/ML platforms and solutions
9/52: Leading technical teams and mentoring engineers
10/52: Cross-functional collaboration and stakeholder alignment
11/52: Translating business requirements into technical solutions
12/52: Evaluating, integrating, and operationalizing emerging AI technologies (GenAI, agentic AI, RAG)
13/52: Establishing and enforcing governance, compliance, and responsible AI practices
14/52: Optimizing system performance, cost, and reliability
15/

In [15]:
def clean_signal_evidence_map(signal_evidence_map, candidate_evidence, min_relevance=3):
    valid_evidence_ids = {e["evidence_id"] for e in candidate_evidence}

    cleaned = []
    removed = []

    for signal_item in signal_evidence_map:
        kept_matches = []

        for match in signal_item.get("supporting_evidence", []):
            evidence_id = match.get("evidence_id")
            relevance = match.get("relevance", 0)

            if evidence_id not in valid_evidence_ids:
                removed.append({
                    "signal_id": signal_item.get("signal_id"),
                    "signal": signal_item.get("signal"),
                    "reason_removed": "invalid_evidence_id",
                    "match": match
                })
                continue

            if relevance < min_relevance:
                removed.append({
                    "signal_id": signal_item.get("signal_id"),
                    "signal": signal_item.get("signal"),
                    "reason_removed": "below_min_relevance",
                    "match": match
                })
                continue

            kept_matches.append(match)

        cleaned_item = {
            **signal_item,
            "supporting_evidence": kept_matches,
        }

        cleaned.append(cleaned_item)

    return cleaned, removed

In [16]:
signal_evidence_map_cleaned, removed_matches = clean_signal_evidence_map(
    signal_evidence_map,
    candidate_evidence,
    min_relevance=3,
)

len(removed_matches), removed_matches[:5]

(0, [])

In [17]:
len(signal_evidence_map), len(mapping_errors)

(52, 0)

In [18]:
save_json(signal_evidence_map, ARTIFACT_DIR / "signal_evidence_map.json")


In [19]:
from collections import Counter

Counter(item["coverage_assessment"] for item in signal_evidence_map)

Counter({'strong': 33, 'moderate': 18, 'unsupported': 1})

In [20]:
for item in signal_evidence_map:
    print(
        item["coverage_assessment"].upper(),
        "|",
        item["source_field"],
        "|",
        item["signal"],
        "| evidence:",
        len(item.get("supporting_evidence", []))
    )

MODERATE | core_market_themes | Enterprise-scale AI/ML platform architecture and delivery | evidence: 3
MODERATE | core_market_themes | Productionization and operationalization of generative and agentic AI | evidence: 3
MODERATE | core_market_themes | Cloud-native, scalable, and resilient system design | evidence: 3
STRONG | core_market_themes | MLOps, LLMOps, and observability as operational foundations | evidence: 4
MODERATE | core_market_themes | Governance, compliance, and responsible AI as non-negotiable requirements | evidence: 3
STRONG | core_market_themes | Mentorship, cross-functional leadership, and technical influence | evidence: 3
STRONG | core_market_themes | Continuous innovation, research application, and developer enablement | evidence: 3
STRONG | target_capabilities | Architecting and designing scalable AI/ML platforms and solutions | evidence: 3
STRONG | target_capabilities | Leading technical teams and mentoring engineers | evidence: 3
STRONG | target_capabilities | 

In [21]:
evidence_by_id = {e["evidence_id"]: e for e in candidate_evidence}

for signal_item in signal_evidence_map:
    print("\n" + "=" * 100)
    print("SIGNAL:", signal_item["signal"])
    print("FIELD:", signal_item["source_field"])
    print("COVERAGE:", signal_item["coverage_assessment"])
    print("REASON:", signal_item["coverage_reason"])

    for match in signal_item.get("supporting_evidence", []):
        ev = evidence_by_id.get(match["evidence_id"])

        print("\n ", match["relevance"], match["support_type"])
        print(" ", ev.get("source_label", ""))
        print(" ", ev["evidence_text"])
        print(" ", "Reason:", match["reason"])


SIGNAL: Enterprise-scale AI/ML platform architecture and delivery
FIELD: core_market_themes
COVERAGE: moderate
REASON: There are two direct, credible evidence items describing the architecture and delivery of enterprise-scale AI/ML or analytics platforms using modern cloud and governance patterns, plus one partial match for large-scale platform delivery. No evidence describes end-to-end ownership of a full AI/ML platform, but the provided items demonstrate substantial relevant experience.

  4 direct
  Daly Engineers LLC | Principal Consultant | Consolidated Edison (via ABC Consulting)
  Built a governed analytics assistant using Gemini, BigQuery, and tool-calling architectures that applied analyst rules and operational controls while maintaining predictable and auditable execution.
  Reason: Describes building a governed analytics assistant using Gemini, BigQuery, and tool-calling architectures, with operational controls and auditable execution—demonstrating direct architecture and d

In [22]:
valid_evidence_ids = {e["evidence_id"] for e in candidate_evidence}

bad_matches = []

for signal_item in signal_evidence_map:
    for match in signal_item.get("supporting_evidence", []):
        evidence_id = match.get("evidence_id")
        if evidence_id not in valid_evidence_ids:
            bad_matches.append({
                "signal_id": signal_item.get("signal_id"),
                "signal": signal_item.get("signal"),
                "bad_evidence_id": evidence_id,
                "match": match
            })

bad_matches

[]

In [23]:
import statistics
from collections import Counter

counts = [
    len(item.get("supporting_evidence", []))
    for item in signal_evidence_map_cleaned
]

print("coverage:", Counter(item["coverage_assessment"] for item in signal_evidence_map_cleaned))
print("min:", min(counts))
print("median:", statistics.median(counts))
print("mean:", round(statistics.mean(counts), 2))
print("max:", max(counts))

coverage: Counter({'strong': 33, 'moderate': 18, 'unsupported': 1})
min: 0
median: 3.0
mean: 3.1
max: 5


In [32]:
def recompute_coverage_assessment(supporting_evidence, evidence_by_id):
    """
    Conservative deterministic coverage label.

    Requires both evidence strength and source diversity.
    """
    if not supporting_evidence:
        return "unsupported"

    direct_4_plus = [
        e for e in supporting_evidence
        if e.get("support_type") == "direct" and e.get("relevance", 0) >= 4
    ]

    direct_3_plus = [
        e for e in supporting_evidence
        if e.get("support_type") == "direct" and e.get("relevance", 0) >= 3
    ]

    partial_3_plus = [
        e for e in supporting_evidence
        if e.get("support_type") == "partial" and e.get("relevance", 0) >= 3
    ]

    source_labels = {
        evidence_by_id[e["evidence_id"]].get("source_label", "")
        for e in supporting_evidence
        if e.get("evidence_id") in evidence_by_id
    }

    direct_4_sources = {
        evidence_by_id[e["evidence_id"]].get("source_label", "")
        for e in direct_4_plus
        if e.get("evidence_id") in evidence_by_id
    }

    # Strong should be hard to earn:
    # multiple strong direct examples from more than one source.
    if len(direct_4_plus) >= 3 and len(direct_4_sources) >= 2:
        return "strong"

    # Moderate: at least one strong direct example,
    # or multiple direct examples,
    # or several partial examples across sources.
    if len(direct_4_plus) >= 1:
        return "moderate"

    if len(direct_3_plus) >= 2 and len(source_labels) >= 2:
        return "moderate"

    if len(partial_3_plus) >= 3 and len(source_labels) >= 2:
        return "moderate"

    return "weak"

In [34]:
evidence_by_id = {e["evidence_id"]: e for e in candidate_evidence}

for item in signal_evidence_map:
    item["model_coverage_assessment"] = item.get("model_coverage_assessment", item.get("coverage_assessment"))
    item["coverage_assessment"] = recompute_coverage_assessment(
        item.get("supporting_evidence", []),
        evidence_by_id
    )

In [37]:
from collections import Counter
import statistics

counts = [
    len(item.get("supporting_evidence", []))
    for item in signal_evidence_map
]

print("coverage:", Counter(item["coverage_assessment"] for item in signal_evidence_map))
print("model coverage:", Counter(item["model_coverage_assessment"] for item in signal_evidence_map))
print("min:", min(counts))
print("median:", statistics.median(counts))
print("mean:", round(statistics.mean(counts), 2))
print("max:", max(counts))

coverage: Counter({'strong': 30, 'moderate': 21, 'unsupported': 1})
model coverage: Counter({'strong': 33, 'moderate': 18, 'unsupported': 1})
min: 0
median: 3.0
mean: 3.1
max: 5


In [38]:
for item in signal_evidence_map:
    if item["model_coverage_assessment"] != item["coverage_assessment"]:
        print(
            item["model_coverage_assessment"],
            "->",
            item["coverage_assessment"],
            "|",
            item["source_field"],
            "|",
            item["signal"]
        )

strong -> moderate | target_problem_spaces | AI/ML model lifecycle management and deployment
strong -> moderate | target_problem_spaces | Data integration, quality, and interoperability
strong -> moderate | target_problem_spaces | Personalization and customer experience enhancement
strong -> moderate | target_technology_areas | AI observability and monitoring tools
moderate -> strong | success_patterns | Continuous improvement and rapid integration of emerging technologies


In [39]:
with open(ARTIFACT_DIR / "signal_evidence_map_v1.json", "w", encoding="utf-8") as f:
    json.dump(signal_evidence_map, f, indent=2)

## 4C. Score Evidence Against Market Signals

Next: evidence scoring

Now invert the map:

signal -> evidence

into:

evidence -> signal matches

This can be mostly deterministic. No need to ask the LLM to rediscover the same evidence.

In [47]:
def build_evidence_scores(candidate_evidence, signal_evidence_map):
    evidence_by_id = {e["evidence_id"]: e for e in candidate_evidence}

    scored = {
        evidence_id: {
            **evidence,
            "signal_matches": [],
            "weighted_score": 0.0,
            "max_relevance": 0,
            "direct_match_count": 0,
            "matched_signal_count": 0,
        }
        for evidence_id, evidence in evidence_by_id.items()
    }

    support_type_multiplier = {
        "direct": 1.0,
        "partial": 0.7,
        "adjacent": 0.4,
    }

    for signal_item in signal_evidence_map:
        signal_id = signal_item["signal_id"]
        signal = signal_item["signal"]
        source_field = signal_item["source_field"]
        priority_weight = signal_item.get("priority_weight", 1)

        for match in signal_item.get("supporting_evidence", []):
            evidence_id = match.get("evidence_id")

            if evidence_id not in scored:
                continue

            relevance = match.get("relevance", 0)
            support_type = match.get("support_type", "partial")
            multiplier = support_type_multiplier.get(support_type, 0.5)

            contribution = relevance * priority_weight * multiplier

            scored[evidence_id]["signal_matches"].append({
                "signal_id": signal_id,
                "signal": signal,
                "source_field": source_field,
                "priority_weight": priority_weight,
                "relevance": relevance,
                "support_type": support_type,
                "contribution": contribution,
                "reason": match.get("reason", ""),
            })

            scored[evidence_id]["weighted_score"] += contribution
            scored[evidence_id]["max_relevance"] = max(
                scored[evidence_id]["max_relevance"],
                relevance
            )

            if support_type == "direct":
                scored[evidence_id]["direct_match_count"] += 1

            scored[evidence_id]["matched_signal_count"] += 1

    evidence_scores = list(scored.values())

    evidence_scores = sorted(
        evidence_scores,
        key=lambda x: (
            x["weighted_score"],
            x["direct_match_count"],
            x["max_relevance"],
            x["matched_signal_count"],
        ),
        reverse=True
    )

    return evidence_scores

In [48]:
def add_field_capped_score(evidence_scores, cap_per_field=2):
    for item in evidence_scores:
        matches = item.get("signal_matches", [])

        by_field = {}
        for m in matches:
            by_field.setdefault(m["source_field"], []).append(m)

        capped_score = 0.0

        for field, field_matches in by_field.items():
            top_matches = sorted(
                field_matches,
                key=lambda x: x.get("contribution", 0),
                reverse=True
            )[:cap_per_field]

            capped_score += sum(m.get("contribution", 0) for m in top_matches)

        item["field_capped_score"] = capped_score

    return sorted(
        evidence_scores,
        key=lambda x: (
            x["field_capped_score"],
            x["direct_match_count"],
            x["max_relevance"],
            x["matched_signal_count"],
        ),
        reverse=True
    )

In [49]:
evidence_scores = build_evidence_scores(
    candidate_evidence,
    signal_evidence_map,
)
evidence_scores_capped = add_field_capped_score(evidence_scores)

for e in evidence_scores_capped[:15]:
    print(
        round(e["field_capped_score"], 1),
        "| raw:",
        round(e["weighted_score"], 1),
        "| direct:",
        e["direct_match_count"],
        "| signals:",
        e["matched_signal_count"],
        "|",
        e["source_label"]
    )
    print(e["evidence_text"])
    print()

with open(ARTIFACT_DIR / "evidence_scores.json", "w", encoding="utf-8") as f:
    json.dump(evidence_scores_capped, f, indent=2)

len(evidence_scores_capped), evidence_scores_capped[:3]

124.3 | raw: 202.3 | direct: 18 | signals: 19 | Capital One | Senior Manager, Operations Analysis
Conceived, designed, and led adoption of RITHM (Real-Time IT Health Monitoring), an operational intelligence platform that aggregated monitoring signals by business capability and system ownership, reducing major incident detection times from hours to minutes.

121.0 | raw: 175.2 | direct: 16 | signals: 17 | Daly Engineers LLC | Principal Consultant | Consolidated Edison (via ABC Consulting)
Built a governed analytics assistant using Gemini, BigQuery, and tool-calling architectures that applied analyst rules and operational controls while maintaining predictable and auditable execution.

106.0 | raw: 118.2 | direct: 11 | signals: 12 | Daly Engineers LLC | Principal Consultant | NBCUniversal (via Apex Systems)
Built AWS-based AI and graph analytics pipelines using SageMaker, Terraform, S3, and containerized workloads.

104.0 | raw: 112.0 | direct: 10 | signals: 10 | Meta | Data Scientist
De

(46,
 [{'evidence_id': 'exp_01_09_03_01',
   'source_label': 'Capital One | Senior Manager, Operations Analysis',
   'source_type': 'experience_bullet',
   'source_section': 'Professional Experience',
   'organization': 'Capital One',
   'role': 'Senior Manager, Operations Analysis',
   'dates': 'Jan 2012 - Dec 2016',
   'label': '',
   'context': '',
   'evidence_text': 'Conceived, designed, and led adoption of RITHM (Real-Time IT Health Monitoring), an operational intelligence platform that aggregated monitoring signals by business capability and system ownership, reducing major incident detection times from hours to minutes.',
   'raw_text': 'Conceived, designed, and led adoption of RITHM (Real-Time IT Health Monitoring), an operational intelligence platform that aggregated monitoring signals by business capability and system ownership, reducing major incident detection times from hours to minutes.',
   'confidence': 'high',
   'signal_matches': [{'signal_id': 'core_market_themes_00

In [51]:
for e in evidence_scores_capped[:15]:
    print("\n" + "=" * 100)
    print(e["weighted_score"], "| direct:", e["direct_match_count"], "| signals:", e["matched_signal_count"])
    print(e["source_label"])
    print(e["evidence_text"])

    top_matches = sorted(
        e["signal_matches"],
        key=lambda m: m["contribution"],
        reverse=True
    )[:5]

    for m in top_matches:
        print("  -", m["relevance"], m["support_type"], "|", m["signal"])


202.3 | direct: 18 | signals: 19
Capital One | Senior Manager, Operations Analysis
Conceived, designed, and led adoption of RITHM (Real-Time IT Health Monitoring), an operational intelligence platform that aggregated monitoring signals by business capability and system ownership, reducing major incident detection times from hours to minutes.
  - 5 direct | MLOps, LLMOps, and observability as operational foundations
  - 5 direct | Translating business requirements into technical solutions
  - 5 direct | Strategic and visionary leadership (roadmap, technical vision, innovation)
  - 5 direct | Ownership of delivery, architecture, and operational outcomes
  - 4 direct | Architecting and designing scalable AI/ML platforms and solutions

175.2 | direct: 16 | signals: 17
Daly Engineers LLC | Principal Consultant | Consolidated Edison (via ABC Consulting)
Built a governed analytics assistant using Gemini, BigQuery, and tool-calling architectures that applied analyst rules and operational cont

In [53]:
save_json(evidence_scores_capped, ARTIFACT_DIR / "evidence_scores.json")

## 4D. Select Evidence for Target Resume

### Purpose

Convert scored evidence into a smaller set of resume-worthy evidence items.

This step decides which accomplishments deserve resume space for the target archetype. It does not rewrite bullets and does not generate the final resume.

### Inputs

- `artifacts/candidate_evidence.json`
  - Extracted evidence units from the canonical resume.
- `artifacts/signal_evidence_map_v1.json`
  - Market signal to evidence mapping.
- `artifacts/evidence_scores.json`
  - Evidence-first scoring output with raw and capped alignment scores.
- `artifacts/target_archetype.json`
  - Target role archetype and market-derived priorities.

### Output

- `artifacts/selected_evidence.json`

### Selection Question

For each evidence item:

> Does this accomplishment deserve resume space for the target archetype?

### Selection Factors

Use the evidence score as the starting point, then apply judgment for:

- Alignment with the target archetype
- Strength of direct evidence
- Recency
- Role/source diversity
- Avoiding redundant proof points
- Avoiding overstatement
- Preserving career depth where it strengthens the target story

### Output Shape

```json
{
  "selection_summary": {
    "target_archetype": "Principal AI Architect / AI Platform Engineering Lead",
    "total_evidence_items": 46,
    "selected_count": 0,
    "selection_method": "field_capped_score plus qualitative review"
  },
  "selected_evidence": [
    {
      "evidence_id": "exp_01_01_01_04",
      "source_label": "Daly Engineers LLC | Principal Consultant | Consolidated Edison (via ABC Consulting)",
      "evidence_text": "...",
      "selection_tier": "must_include",
      "selection_reason": "Current, direct evidence of governed GenAI analytics architecture with tool-calling, BigQuery, controls, and auditable execution.",
      "field_capped_score": 121.0,
      "direct_match_count": 16,
      "primary_supporting_signals": [
        "Productionization and operationalization of generative and agentic AI",
        "Governance, compliance, and responsible AI as non-negotiable requirements",
        "Architecting and designing scalable AI/ML platforms and solutions"
      ],
      "cautions": [
        "Avoid implying full enterprise production deployment unless supported elsewhere."
      ]
    }
  ],
  "excluded_evidence": [
    {
      "evidence_id": "...",
      "selection_tier": "exclude",
      "selection_reason": "Lower target relevance or redundant with stronger evidence."
    }
  ]
}

**Selection Tiers**
- must_include     = essential to target story
- strong_include   = strong evidence, likely included if space allows
- optional         = useful but not necessary
- compress         = combine with nearby evidence or mention briefly
- exclude          = do not use for this target resume

**Completion Criteria**

4D is complete when:
- selected_evidence.json exists
- Each selected item has a clear selection tier
- The cut line is explicit
- Redundant evidence is identified
- Older evidence is included only when it strengthens the target story
- No selected item depends on exaggerated interpretation

In [62]:
evidence_scores = load_json(ARTIFACT_DIR / "evidence_scores.json")
target_archetype = load_json(ARTIFACT_DIR / "target_archetype.json")

In [63]:
CANDIDATE_SPECIFIC_CAUTION_RULES = [
    {
        "name": "teaching_not_production_ownership",
        "match_field": "source_label",
        "contains_any": ["AI Instructor"],
        "caution": (
            "Teaching evidence supports mentorship, technical fluency, and developer enablement; "
            "do not present it as production ownership unless the bullet explicitly supports that."
        ),
    },
    {
        "name": "older_evidence_supports_depth",
        "match_field": "source_label",
        "contains_any": ["Capital One", "Raytheon"],
        "caution": (
            "Older evidence; use as career-depth proof rather than the lead current AI story."
        ),
    },
    {
        "name": "nbcu_short_engagement",
        "match_field": "source_label",
        "contains_any": ["NBCUniversal"],
        "caution": (
            "Frame carefully as short-cycle architecture/prototype evidence unless additional facts support production deployment."
        ),
    },
    {
        "name": "strategy_not_implementation",
        "match_field": "evidence_text",
        "contains_any": ["roadmap", "assessment"],
        "caution": (
            "Strategy and readiness evidence; do not overstate as implementation unless paired with delivery evidence."
        ),
    },
]

def top_supporting_signals(evidence_item, n=5):
    matches = evidence_item.get("signal_matches", [])

    top = sorted(
        matches,
        key=lambda m: m.get("contribution", 0),
        reverse=True
    )[:n]

    return [
        {
            "signal_id": m.get("signal_id"),
            "signal": m.get("signal"),
            "source_field": m.get("source_field"),
            "relevance": m.get("relevance"),
            "support_type": m.get("support_type"),
            "contribution": m.get("contribution"),
        }
        for m in top
    ]


def get_primary_signal_names(evidence_item, n=3):
    return [
        m["signal"]
        for m in top_supporting_signals(evidence_item, n=n)
    ]


def infer_cautions(evidence_item, caution_rules=CANDIDATE_SPECIFIC_CAUTION_RULES):
    """
    CANDIDATE_SPECIFIC HEURISTIC.

    Applies interpretation guardrails for Doug Daly's current resume evidence set.

    This is real behavior, not a STUB. However, it is intentionally hardwired
    to known risks in this candidate's resume, such as teaching evidence,
    older platform evidence, short-cycle NBCUniversal work, and roadmap/assessment
    language.

    TECH_DEBT:
    For a reusable resume generator, replace this with candidate-specific
    configuration loaded from a separate profile, strategy file, or review artifact.
    """
    cautions = []

    for rule in caution_rules:
        field_value = evidence_item.get(rule["match_field"], "")

        if not isinstance(field_value, str):
            continue

        field_value_lower = field_value.lower()

        if any(term.lower() in field_value_lower for term in rule["contains_any"]):
            cautions.append(rule["caution"])

    return cautions

In [64]:
def assign_selection_tier(evidence_item, rank):
    score = evidence_item.get("field_capped_score", evidence_item.get("weighted_score", 0))
    direct = evidence_item.get("direct_match_count", 0)
    matches = evidence_item.get("matched_signal_count", 0)

    if matches == 0 or score == 0:
        return "exclude"

    # Essential evidence for the target story.
    if rank <= 6 and direct >= 4:
        return "must_include"

    # Strong evidence likely worth resume space.
    if rank <= 15 and direct >= 3:
        return "strong_include"

    # Useful evidence, but may lose space in a tight resume.
    if rank <= 25 and direct >= 2:
        return "optional"

    # Evidence has value but should be combined, compressed, or used only if needed.
    if matches > 0:
        return "compress"

    return "exclude"

In [65]:
def build_selected_evidence(evidence_scores, target_archetype):
    sorted_evidence = sorted(
        evidence_scores,
        key=lambda e: (
            e.get("field_capped_score", e.get("weighted_score", 0)),
            e.get("direct_match_count", 0),
            e.get("max_relevance", 0),
            e.get("matched_signal_count", 0),
        ),
        reverse=True
    )

    selected_items = []
    excluded_items = []

    for rank, item in enumerate(sorted_evidence, start=1):
        tier = assign_selection_tier(item, rank)

        base_record = {
            "evidence_id": item.get("evidence_id"),
            "rank": rank,
            "selection_tier": tier,
            "source_label": item.get("source_label", ""),
            "source_type": item.get("source_type", ""),
            "organization": item.get("organization", ""),
            "role": item.get("role", ""),
            "dates": item.get("dates", ""),
            "label": item.get("label", ""),
            "evidence_text": item.get("evidence_text", ""),
            "field_capped_score": item.get("field_capped_score"),
            "weighted_score": item.get("weighted_score"),
            "direct_match_count": item.get("direct_match_count"),
            "matched_signal_count": item.get("matched_signal_count"),
            "max_relevance": item.get("max_relevance"),
            "primary_supporting_signals": get_primary_signal_names(item, n=5),
            "top_signal_matches": top_supporting_signals(item, n=5),
            "cautions": infer_cautions(item),
        }

        if tier == "exclude":
            base_record["selection_reason"] = (
                "Excluded for this target because it has weak or no meaningful alignment "
                "with the market-derived target signals."
            )
            excluded_items.append(base_record)
        else:
            base_record["selection_reason"] = (
                f"Selected as {tier} based on field-capped alignment score "
                f"{round(item.get('field_capped_score', 0), 1)}, "
                f"{item.get('direct_match_count', 0)} direct signal matches, "
                f"and support for primary signals including: "
                f"{'; '.join(get_primary_signal_names(item, n=3))}."
            )
            selected_items.append(base_record)

    summary = {
        "target_archetype": target_archetype.get("title", ""),
        "total_evidence_items": len(evidence_scores),
        "selected_count": len(selected_items),
        "excluded_count": len(excluded_items),
        "selection_method": (
            "Deterministic first-pass selection using field_capped_score, direct_match_count, "
            "rank, and qualitative caution flags. This step selects evidence, not final resume bullets."
        ),
        "tier_counts": {
            tier: sum(1 for x in selected_items + excluded_items if x["selection_tier"] == tier)
            for tier in ["must_include", "strong_include", "optional", "compress", "exclude"]
        }
    }

    return {
        "selection_summary": summary,
        "selected_evidence": selected_items,
        "excluded_evidence": excluded_items,
    }

In [66]:
selected_evidence = build_selected_evidence(
    evidence_scores=evidence_scores,
    target_archetype=target_archetype,
)

with open(ARTIFACT_DIR / "selected_evidence.json", "w", encoding="utf-8") as f:
    json.dump(selected_evidence, f, indent=2)

selected_evidence["selection_summary"]

{'target_archetype': 'Principal AI Architect / AI Platform Engineering Lead',
 'total_evidence_items': 46,
 'selected_count': 39,
 'excluded_count': 7,
 'selection_method': 'Deterministic first-pass selection using field_capped_score, direct_match_count, rank, and qualitative caution flags. This step selects evidence, not final resume bullets.',
 'tier_counts': {'must_include': 6,
  'strong_include': 9,
  'optional': 8,
  'compress': 16,
  'exclude': 7}}

In [67]:
for item in selected_evidence["selected_evidence"]:
    print("\n" + "=" * 100)
    print(
        item["rank"],
        "|",
        item["selection_tier"],
        "| capped:",
        round(item["field_capped_score"], 1),
        "| direct:",
        item["direct_match_count"],
    )
    print(item["source_label"])
    print(item["evidence_text"])

    print("Signals:")
    for signal in item["primary_supporting_signals"][:3]:
        print("  -", signal)

    if item["cautions"]:
        print("Cautions:")
        for caution in item["cautions"]:
            print("  -", caution)


1 | must_include | capped: 124.3 | direct: 18
Capital One | Senior Manager, Operations Analysis
Conceived, designed, and led adoption of RITHM (Real-Time IT Health Monitoring), an operational intelligence platform that aggregated monitoring signals by business capability and system ownership, reducing major incident detection times from hours to minutes.
Signals:
  - MLOps, LLMOps, and observability as operational foundations
  - Translating business requirements into technical solutions
  - Strategic and visionary leadership (roadmap, technical vision, innovation)
Cautions:
  - Older evidence; use as career-depth proof rather than the lead current AI story.

2 | must_include | capped: 121.0 | direct: 16
Daly Engineers LLC | Principal Consultant | Consolidated Edison (via ABC Consulting)
Built a governed analytics assistant using Gemini, BigQuery, and tool-calling architectures that applied analyst rules and operational controls while maintaining predictable and auditable execution.
S

In [68]:
selected_evidence["selection_summary"]["tier_counts"]

{'must_include': 6,
 'strong_include': 9,
 'optional': 8,
 'compress': 16,
 'exclude': 7}

In [69]:
save_json(selected_evidence, ARTIFACT_DIR / "selected_evidence.json")

## Stage 4 Completion Summary

Stage 4 converts the canonical resume into market-aligned evidence.

Outputs:

- `candidate_evidence.json`
- `signal_evidence_map_v1.json`
- `evidence_scores.json`
- `selected_evidence.json`

Key decisions:

- Evidence is scored against market-derived signals, not hardcoded resume themes.
- Signal mapping is conservative and evidence-bounded.
- Final selection uses capped scores plus caution flags to avoid overstatement.
- Selected evidence is not yet resume copy. It is the content foundation for later stages.